# 5 — Dynamic Graphs with edge-of-the-world: Solutions

This file contains worked solutions for all 10 exercises in notebook 5.

**Run notebook 5 first** (or run the setup cells below) before attempting any exercises, as they depend on the nodes, edges, and LCIA method created there.

## Setup

Re-create the complete bicycle eotw database so every solution cell works in isolation. This is identical to the setup in notebook 5.

In [ ]:
import bw2data as bd
import bw2calc as bc
import bw_eotw
from bw_eotw import RichEdge, RichEdges, RichEdgesBackend, RichNode
from bw_eotw import Interpreter, register, MatrixEntry
import pandas as pd

# Clean slate
try:
    bd.projects.delete_project("eotw-bicycle", delete_dir=True)
except ValueError:
    pass

bd.projects.set_current("eotw-bicycle")

In [ ]:
# ── Database ───────────────────────────────────────────────────────────────
db = bd.Database("bicycle", backend="eotw")
db.register()

# ── Nodes ──────────────────────────────────────────────────────────────────
co2_flow = db.new_node(
    code="co2", name="Carbon Dioxide",
    categories=("air",), unit="kilogram",
    type=bd.labels.biosphere_node_default,
)
co2_flow.save()

ng_process = db.new_node(
    code="ng-process", name="natural gas production",
    location="NO", type=bd.labels.process_node_default,
)
ng_product = db.new_node(
    code="ng-product", name="natural gas",
    unit="megajoule", type=bd.labels.product_node_default,
)
ng_process.save(); ng_product.save()

cf_process = db.new_node(
    code="cf-process", name="carbon fibre production",
    location="DE", type=bd.labels.process_node_default,
)
cf_product = db.new_node(
    code="cf-product", name="carbon fibre",
    unit="kilogram", type=bd.labels.product_node_default,
)
cf_process.save(); cf_product.save()

bike_process = db.new_node(
    code="bike-process", name="bike production",
    location="DK", type=bd.labels.process_node_default,
)
bike_product = db.new_node(
    code="bike-product", name="bicycle",
    unit="number", type=bd.labels.product_node_default,
)
bike_process.save(); bike_product.save()

wind_process = db.new_node(
    code="wind-process", name="Danish offshore wind",
    location="DK", type=bd.labels.process_node_default,
)
wind_product = db.new_node(
    code="wind-product", name="electricity, wind",
    unit="kilowatt hour", type=bd.labels.product_node_default,
)
wind_process.save(); wind_product.save()

ng_power_process = db.new_node(
    code="ng-power-process", name="Danish natural gas power",
    location="DK", type=bd.labels.process_node_default,
)
ng_power_product = db.new_node(
    code="ng-power-product", name="electricity, natural gas",
    unit="kilowatt hour", type=bd.labels.product_node_default,
)
ng_power_process.save(); ng_power_product.save()

print("All nodes created:", len(list(db)))

In [ ]:
# ── Edges ──────────────────────────────────────────────────────────────────
# Production edges
for proc, prod in [
    (ng_process, ng_product),
    (cf_process, cf_product),
    (bike_process, bike_product),
    (wind_process, wind_product),
    (ng_power_process, ng_power_product),
]:
    proc.new_edge(
        input=prod, amount=1,
        type=bd.labels.production_edge_default,
        functional=True,
    ).save()

# Static supply chain edges
cf_process.new_edge(
    input=ng_product, amount=237.3,
    type=bd.labels.consumption_edge_default,
).save()

# Biosphere: CF production emits CO2
cf_process.new_edge(
    input=co2_flow, amount=26.6,
    type=bd.labels.biosphere_edge_default,
).save()

# provider_mix: bike_process consumes 100 kWh electricity (wind 70%, gas 30%)
electricity_edge = bike_process.new_edge(
    input=wind_product, amount=100,
    type=bd.labels.consumption_edge_default,
    interpreter="provider_mix",
    product_name="electricity",
    mix=[
        {"input": wind_product,     "share": 0.70},
        {"input": ng_power_product, "share": 0.30},
    ],
)
electricity_edge.save()

# scenario: bike_process consumes CF — scenario-dependent amount
cf_edge = bike_process.new_edge(
    input=cf_product,
    type=bd.labels.consumption_edge_default,
    interpreter="scenario",
    scenario_values={
        "baseline":   2.5,
        "optimistic": 1.5,
        "pessimistic": 3.5,
    },
)
cf_edge.save()

# temporal: ng_power emits CO2 — time-varying intensity
co2_ng_power_edge = ng_power_process.new_edge(
    input=co2_flow,
    type=bd.labels.biosphere_edge_default,
    interpreter="temporal",
    temporal_values={2020: 0.45, 2025: 0.35, 2030: 0.22},
    default_year=2025,
)
co2_ng_power_edge.save()

# LCIA method
ipcc = bd.Method(("IPCC", "GWP100"))
ipcc.write([(co2_flow.key, {"amount": 1.0})])

print("Setup complete")

## Exercise 1: How does the eotw backend differ from `SQLiteBackend`?

**Question:** Check `type(db)`, try `db.write({})`, and explain why `write()` is disabled.

In [ ]:
# type(db) → RichEdgesBackend (a subclass of SQLiteBackend)
print(type(db))
print(type(db).__mro__)  # shows the inheritance chain

try:
    db.write({})
except NotImplementedError as e:
    print("\nwrite() raises NotImplementedError:", e)

**Answer:** `RichEdgesBackend` is a thin subclass of `SQLiteBackend`. It overrides `write()` and `_efficient_write_many_data()` with stubs that raise `NotImplementedError`.

`write()` is disabled because it expects a plain dict of the form `{(db, code): {data}}` and would silently lose the rich `interpreter` keys embedded in edge data. Instead, you build the database node-by-node with `db.new_node()` and `node.new_edge()`, which ensures that `normalize_edge()` and `validate_edge()` run for every edge before it reaches the database.

## Exercise 2: Verify `row` IDs from `provider_mix`

**Question:** What are the `row` IDs for the two MatrixEntry objects? Do they match `wind_product.id` and `ng_power_product.id`?

In [ ]:
entries = electricity_edge.resolve(config={})
print("wind_product.id      =", wind_product.id)
print("ng_power_product.id  =", ng_power_product.id)
print()
for e in entries:
    label = "wind_product" if e.row == wind_product.id else "ng_power_product"
    print(f"  MatrixEntry row={e.row} ({label}), col={e.col}, amount={e.amount:.1f}")

**Answer:** Yes — `row` for the first entry is `wind_product.id` (70 kWh), and for the second is `ng_power_product.id` (30 kWh). The `col` is `bike_process.id` in both cases.

This is the core behaviour of `provider_mix`: the single graph edge becomes one technosphere matrix column entry per provider, each with `amount = total_amount × share`.

## Exercise 3: Add a nuclear provider

**Question:** Add a third electricity source with 10% share. Adjust the other shares so they still sum to 1.0.

In [ ]:
# Create nuclear nodes
nuclear_process = db.new_node(
    code="nuclear-process", name="Danish nuclear (hypothetical)",
    location="DK", type=bd.labels.process_node_default,
)
nuclear_product = db.new_node(
    code="nuclear-product", name="electricity, nuclear",
    unit="kilowatt hour", type=bd.labels.product_node_default,
)
nuclear_process.save(); nuclear_product.save()
nuclear_process.new_edge(
    input=nuclear_product, amount=1,
    type=bd.labels.production_edge_default,
    functional=True,
).save()

# Update the mix — shares must sum exactly to 1.0
# 0.70 - 0.07 = 0.63 (wind), 0.30 - 0.03 = 0.27 (gas), + 0.10 (nuclear) = 1.00
electricity_edge["mix"] = [
    {"input": wind_product,     "share": 0.63},
    {"input": ng_power_product, "share": 0.27},
    {"input": nuclear_product,  "share": 0.10},
]
electricity_edge.save()

entries = electricity_edge.resolve(config={})
print(f"Number of MatrixEntry objects: {len(entries)}")
for e in entries:
    print(f"  row={e.row}, amount={e.amount:.1f} kWh")
total_share = sum(e.amount for e in entries) / 100
print(f"Shares sum to: {total_share:.2f}")

**Answer:** After updating `electricity_edge["mix"]` with three providers (wind 63%, gas 27%, nuclear 10%) and calling `save()`, `resolve()` returns three `MatrixEntry` objects — one per provider. The amounts are 63, 27, and 10 kWh, summing to 100 kWh. The `save()` call validates that shares sum to 1.0, so any mistake is caught immediately.

## Exercise 4: Non-zero count before vs after provider_mix

**Question:** Why does adding a single provider_mix edge increase the non-zero count in the technosphere matrix by more than 1?

In [ ]:
with db.set_config({}):
    db.process()
    fu_m, data_objs_m, _ = bd.prepare_lca_inputs(
        {bike_product: 1}, remapping=False,
    )

lca_m = bc.LCA(demand=fu_m, data_objs=data_objs_m)
lca_m.lci()
print("Technosphere matrix shape:", lca_m.technosphere_matrix.shape)
print("Technosphere non-zeros:   ", lca_m.technosphere_matrix.nnz)
print("Biosphere non-zeros:      ", lca_m.biosphere_matrix.nnz)

print()
print("Plain bicycle (no electricity):")
print("  3 production edges (ng, cf, bike)")
print("  2 consumption edges (cf->ng, bike->cf)")
print("  subtotal: 5 technosphere non-zeros + 1 biosphere non-zero")
print()
print("After adding electricity (3 providers + provider_mix + CO2 on ng_power):")
print("  +3 production edges (wind, ng_power, nuclear)")
print("  +3 consumption entries from provider_mix (one per provider)")
print("  subtotal: +6 technosphere non-zeros + 1 biosphere non-zero")

**Answer:** A single provider_mix *graph edge* expands into **N technosphere rows** at processing time — one per provider in the `mix` list. In this case N = 3, so the provider_mix edge alone contributes 3 new technosphere non-zeros. On top of that, each new electricity process node needs its own production edge in the diagonal of the matrix (3 more non-zeros), and `ng_power_process` has a biosphere edge (1 more). This is the key demonstration that the graph is richer than the matrix: one edge in the graph can produce multiple entries in the matrix.

## Exercise 5: Add a `"recycled"` scenario

**Question:** Add a fourth scenario with 1.0 kg of carbon fibre. What score do you get?

In [ ]:
# Add the new scenario value in-place and save
cf_edge["scenario_values"]["recycled"] = 1.0
cf_edge.save()

with db.set_config({"scenario": "recycled"}):
    db.process()
    fu, data_objs, _ = bd.prepare_lca_inputs(
        {bike_product: 1},
        method=("IPCC", "GWP100"),
        remapping=False,
    )

lca = bc.LCA(demand=fu, data_objs=data_objs)
lca.lci()
lca.lcia()
print(f"recycled scenario: {lca.score:.2f} kg CO2-eq")
print(f"Expected: 1.0 × 26.6 = {1.0 * 26.6:.2f} kg CO2-eq (from CF)")

**Answer:** The score for the `"recycled"` scenario should be approximately 26.6 kg CO2-eq (1.0 kg CF × 26.6 kg CO2/kg CF), plus a small contribution from the electricity mix. Adding a new scenario requires only modifying `scenario_values` in the existing edge — no new database, no new edges.

## Exercise 6: Is the score-vs-CF relationship linear?

**Question:** Compare the four scenario scores. Is the relationship linear? Why?

In [ ]:
scenario_cf = {
    "recycled":   1.0,
    "optimistic": 1.5,
    "baseline":   2.5,
    "pessimistic": 3.5,
}

print(f"{'Scenario':12s}  {'CF (kg)':>8}  {'Score':>8}  {'Score/CF':>10}")
print("-" * 47)
ratio_vals = []
for name, cf_amount in sorted(scenario_cf.items(), key=lambda x: x[1]):
    with db.set_config({"scenario": name}):
        db.process()
        fu, data_objs, _ = bd.prepare_lca_inputs(
            {bike_product: 1},
            method=("IPCC", "GWP100"),
            remapping=False,
        )
    lca2 = bc.LCA(demand=fu, data_objs=data_objs)
    lca2.lci(); lca2.lcia()
    s = lca2.score
    ratio_vals.append(s / cf_amount)
    print(f"{name:12s}  {cf_amount:>8.1f}  {s:>8.2f}  {s/cf_amount:>10.4f}")

spread = max(ratio_vals) - min(ratio_vals)
print(f"\nSpread in Score/CF ratio: {spread:.6f}")
print("→ Relationship is linear (constant ratio within floating-point precision)")

**Answer:** Yes, the relationship is linear. The Score/CF ratio is the same across all scenarios (within floating-point precision).

The reason follows directly from the matrix equation:

    score = characterisation_vector @ biosphere_matrix @ A^{-1} @ demand_vector

The `scenario` interpreter changes only the **amount** of one technosphere entry (the CF consumption). This scales that column of `A^{-1}` proportionally, which in turn scales the inventory and the score proportionally. There are no non-linearities anywhere in the standard LCA calculation.

## Exercise 7: What happens with an unknown year?

**Question:** Call `co2_ng_power_edge.resolve(config={"year": 2040})`. What error do you get and why?

In [ ]:
try:
    entries = co2_ng_power_edge.resolve(config={"year": 2040})
    print(entries)
except KeyError as e:
    print("KeyError:", e)

**Answer:** A `KeyError` is raised with a message listing the available years. The `temporal` interpreter's resolution order is:

1. Use `config["year"]` if it is in `temporal_values`.
2. Fall back to `edge_data["default_year"]` if it is in `temporal_values`.
3. Raise `KeyError` if neither succeeds.

Year 2040 is not in `temporal_values` (which contains 2020, 2025, 2030), and `default_year=2025` would only be used if no `year` key appeared in `config`. Since `config["year"] = 2040` is present but not in the dict, the interpreter falls through to the `KeyError`. This is a deliberate choice: silent extrapolation outside the defined range would be worse than a clear error.

## Exercise 8: Add year 2035

**Question:** Add CO₂ intensity 0.10 kg/kWh for 2035. What score do you get?

In [ ]:
# Extend the temporal edge in-place
co2_ng_power_edge["temporal_values"][2035] = 0.10
co2_ng_power_edge.save()

with db.set_config({"scenario": "baseline", "year": 2035}):
    db.process()
    fu, data_objs, _ = bd.prepare_lca_inputs(
        {bike_product: 1},
        method=("IPCC", "GWP100"),
        remapping=False,
    )

lca = bc.LCA(demand=fu, data_objs=data_objs)
lca.lci()
lca.lcia()
print(f"Year 2035, baseline scenario: {lca.score:.2f} kg CO2-eq")

# Also print all years for comparison
print()
for year in [2020, 2025, 2030, 2035]:
    with db.set_config({"scenario": "baseline", "year": year}):
        db.process()
        fu, data_objs, _ = bd.prepare_lca_inputs(
            {bike_product: 1},
            method=("IPCC", "GWP100"),
            remapping=False,
        )
    lca2 = bc.LCA(demand=fu, data_objs=data_objs)
    lca2.lci(); lca2.lcia()
    print(f"  Year {year}: {lca2.score:.2f} kg CO2-eq")

**Answer:** The score for 2035 is lower than 2030 because the CO2 intensity of natural gas electricity has dropped further (0.10 vs 0.22 kg CO2/kWh). The trend from 2020 to 2035 shows steady decarbonisation of the electricity supply.

## Exercise 9: All 9 combinations (3 scenarios × 3 years)

**Question:** Which combination gives the lowest impact? Which gives the highest?

In [ ]:
scenarios = ["optimistic", "baseline", "pessimistic"]
years = [2020, 2025, 2030]

results = {}
for scenario in scenarios:
    results[scenario] = {}
    for year in years:
        with db.set_config({"scenario": scenario, "year": year}):
            db.process()
            fu, data_objs, _ = bd.prepare_lca_inputs(
                {bike_product: 1},
                method=("IPCC", "GWP100"),
                remapping=False,
            )
        lca = bc.LCA(demand=fu, data_objs=data_objs)
        lca.lci(); lca.lcia()
        results[scenario][year] = round(lca.score, 2)

df = pd.DataFrame(results, index=years)
df.index.name = "year \ scenario"
print(df.to_string())

print()
flat = [(s, y, v) for s, d in results.items() for y, v in d.items()]
flat.sort(key=lambda x: x[2])
print(f"Lowest:  scenario={flat[0][0]}, year={flat[0][1]}, score={flat[0][2]:.2f} kg CO2-eq")
print(f"Highest: scenario={flat[-1][0]}, year={flat[-1][1]}, score={flat[-1][2]:.2f} kg CO2-eq")

**Answer:** The lowest impact comes from the `optimistic` scenario in 2030 (lightweight carbon fibre + cleaner electricity), and the highest from the `pessimistic` scenario in 2020 (heavy-duty frame + dirtiest electricity grid).

The scenario dimension (CF amount) dominates because the main emission hotspot in this model is CF production (26.6 kg CO2/kg CF × 1.5–3.5 kg), while the electricity contribution is smaller (30 kWh × 0.22–0.45 kg CO2/kWh = 6.6–13.5 kg CO2).

## Exercise 10: LCA score for year 2027 with `linear_trend`

**Question:** Register the `linear_trend` interpreter, add the wind-turbine lifecycle CO2 edge, and compute the score for year 2027 (baseline scenario).

In [ ]:
@register("linear_trend")
class LinearTrendInterpreter(Interpreter):
    """Interpolates linearly between start_value and end_value."""

    def __call__(self, edge_data: dict, config: dict):
        year = config.get("year", edge_data.get("start_year", 2020))
        start_year = edge_data.get("start_year", 2020)
        end_year = edge_data.get("end_year", 2030)
        start = edge_data["start_value"]
        end = edge_data["end_value"]
        t = max(0.0, min(1.0, (year - start_year) / (end_year - start_year)))
        amount = start + t * (end - start)
        yield MatrixEntry(
            row=edge_data["row"], col=edge_data["col"],
            amount=amount, flip=edge_data.get("flip", False),
        )

    def iter_node_ids(self, edge_data: dict):
        yield from ()

    def normalize(self, edge_data: dict) -> None:
        if "amount" not in edge_data:
            edge_data["amount"] = edge_data.get("start_value", 0.0)

    def validate(self, edge_data: dict) -> None:
        if "start_value" not in edge_data or "end_value" not in edge_data:
            raise ValueError("linear_trend requires 'start_value' and 'end_value'")
        super().validate(edge_data)


# Add the linear_trend edge to wind_process
wind_co2_edge = wind_process.new_edge(
    input=co2_flow,
    type=bd.labels.biosphere_edge_default,
    interpreter="linear_trend",
    start_value=0.012,
    end_value=0.006,
    start_year=2020,
    end_year=2030,
)
wind_co2_edge.save()

# Verify the interpolated value at year 2027
# t = (2027 - 2020) / (2030 - 2020) = 7/10 = 0.7
# amount = 0.012 + 0.7 * (0.006 - 0.012) = 0.012 - 0.0042 = 0.0078
e2027 = wind_co2_edge.resolve(config={"year": 2027})
print(f"Wind CO2 at 2027: {e2027[0].amount:.4f} kg CO2/kWh")
print(f"Expected: 0.012 + 0.7*(0.006-0.012) = {0.012 + 0.7*(0.006-0.012):.4f}")

print()

# Run LCA for 2027
with db.set_config({"scenario": "baseline", "year": 2027}):
    db.process()
    fu, data_objs, _ = bd.prepare_lca_inputs(
        {bike_product: 1},
        method=("IPCC", "GWP100"),
        remapping=False,
    )

lca = bc.LCA(demand=fu, data_objs=data_objs)
lca.lci()
lca.lcia()
print(f"Year 2027, baseline scenario: {lca.score:.3f} kg CO2-eq")

**Answer:** At year 2027, the linear interpolation gives:

    t = (2027 - 2020) / (2030 - 2020) = 0.7
    CO2 intensity of wind = 0.012 + 0.7 × (0.006 − 0.012) = 0.0078 kg CO2/kWh

This is combined with the temporal natural-gas electricity CO2 intensity interpolated between the 2025 and 2030 discrete values by the `temporal` interpreter (which picks the nearest defined year — 2025, since 2027 is not in `temporal_values`). The total score is therefore slightly higher than the 2030 result.

**Key take-away:** `linear_trend` is a custom interpreter that lives alongside the built-in ones — the registration mechanism is public and intentionally simple.